In [ ]:
import sys
print(sys.prefix)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import datetime
# %matplotlib notebook
%matplotlib inline

In [ ]:
matplotlib.__version__

In [ ]:
def get_timedelta(duration_h_m_s):
    try:
        t_parse = datetime.datetime.strptime(duration_h_m_s,"%H:%M:%S")
    except:
        sys.stderr.write("Error parsing: '{}'".format(duration_h_m_s))
        raise
    t_delta = datetime.timedelta(hours=t_parse.hour, minutes=t_parse.minute, seconds=t_parse.second)
    return t_delta

In [ ]:
df_tsv = pd.read_csv("Fellowship of the Ring - page-to-timestamp.tsv",sep='\t')

In [ ]:
df_tsv

In [ ]:
df_chapters = pd.read_csv("Fellowship of the Ring - chapters.tsv",sep='\t')
df_chapters

In [ ]:
def get_duration_s(maybe_str):
    if maybe_str is not np.nan:
        t_delta = get_timedelta(maybe_str)
        t_s = t_delta.total_seconds()
    else:
        t_s = np.nan
    return t_s
page_number_raw = df_tsv['Cumulative page number']
t_start_str = df_tsv['Movie start']
t_stop_str = df_tsv['Movie stop']
page_number_l = []
t_start_l = []
t_stop_l = []
for page, t1, t2 in zip(page_number_raw, t_start_str, t_stop_str):
    page_number_l.append(page)
    t_start_l.append(get_duration_s(t1))
    t_stop_l.append(get_duration_s(t2))
page_number = np.array(page_number_l)
t_start = np.array(t_start_l)
t_stop = np.array(t_stop_l)
t_mid = (t_start + t_stop)/2
df_plot = pd.DataFrame(data={
    'page_number':page_number,
    't_start':t_start,
    't_stop':t_stop,
    't_mid':t_mid,
})

In [ ]:
df_tsv['t_mid'] = t_mid
df_tsv['page'] = df_tsv['Cumulative page number']

In [ ]:
page_number = df_tsv['Cumulative page number']
t_start = pd.to_datetime(df_tsv['Movie start'], format='%H:%M:%S').dt.time
t_stop = pd.to_datetime(df_tsv['Movie stop'], format='%H:%M:%S').dt.time
# t_interval = pd.Interval(left=t_start, right=t_stop, closed='both')
# t_mid = t_interval.mid
# df_plot = pd.DataFrame(data={
#     'page_number':page_number,
#     't_start':t_start,
#     't_stop':t_stop,
#     't_mid':t_mid,
# })

In [ ]:
chapter_page_numbers = df_chapters.Page.to_list()

In [ ]:
t_start.shape, df_plot.page_number.shape

In [ ]:
def format_timedelta(x, pos):
    # str(datetime.timedelta) is HH:MM:SS
    return str(datetime.timedelta(seconds=int(x)))

In [ ]:
class QuoteInfo(object):
    def __init__(self, page, text, xy, xytext, fontsize):
        self.page = page
        self.text = text
        self.xy = xy
        self.xytext = xytext
        self.fontsize = fontsize
    def __repr__(self):
        return self.__class__.__name__ + '(' + str(list(self.__dict__.keys())) + ')'
    def __str__(self):
        return self.__class__.__name__ + '(' + str(list(self.__dict__.keys())) + ')'

chosen_i = (4,15,33,47,64,76,93,100,191,208,223,236,245,260,274,302,311,322,344,374,402,418,447,460,478,486,503,513)
quotes = {}
for i_raw in chosen_i:
    i = i_raw - 2
    x = df_tsv.iloc[i].t_mid
    y = df_tsv.iloc[i].page
    quotes[i] = QuoteInfo(
        page = df_tsv.iloc[i].page,
        text = '"' + df_tsv.iloc[i]["Movie quote / description"] + '"',
        xy = (x + 150, y),
        xytext = (x + 800, y + 4),
        fontsize = 8,
    )

In [ ]:
vars(quotes[45])

In [ ]:
fig, ax = plt.subplots(constrained_layout=True, figsize=(6,13), dpi=150)
ax.plot(df_plot.t_mid, df_plot.page_number, '.', color="forestgreen", fillstyle="none", markersize=10);
# yfmt = matplotlib.dates.DateFormatter('%H:%M:%S')
# ax.yaxis.set_major_formatter(yfmt)
# fig.autofmt_ydate();
for index, row in df_chapters.iterrows():
    ax.axhline(y=row.Page, linestyle='--', color='lightgray', zorder=0)
    if row.Book < 2:
        this_x = 12500
        horizontalalignment='right'
    else:
        this_x = 0
        horizontalalignment='left'
    ax.text(this_x, row.Page+6, row.Title, horizontalalignment=horizontalalignment)
    # print(index, row.Title, row.Page)
# Tweak text and positions
quotes[4-2].text = '"Hobbits must seem of little importance"'
quotes[33-2].text = 'Dragon firework'
i = 64-2
quotes[i] = QuoteInfo(
    df_tsv.iloc[i].page,
    '"I wish the Ring had never come to me."',
    (df_tsv.iloc[i].t_mid-150, 55), (df_tsv.iloc[i].t_mid-5100, 55-3), 7)
del i
i = 76-2
quotes[i] = QuoteInfo(
    df_tsv.iloc[i].page,
    '"Many that live deserve death,\nand some that die, deserve life.\nCan you give it to them, Frodo?"',
    (7850, 65), (8500,65+6), 7)
del i
i = 93-2
quotes[i] = QuoteInfo(
    df_tsv.iloc[i].page,
    '"It\'s a dangerous business going out your door."',
    (df_tsv.iloc[i].t_mid+150, df_tsv.iloc[i].page-1),
    (df_tsv.iloc[i].t_mid+800, df_tsv.iloc[i].page-4), 7)
del i
quotes[191-2].fontsize = 7
i = 208-2
quotes[i] = QuoteInfo(
    df_tsv.iloc[i].page,
    '"I think a servant of the Enemy\nwould look fairer, but feel fouler."',
    (df_tsv.iloc[i].t_mid+150, df_tsv.iloc[i].page),
    (df_tsv.iloc[i].t_mid+800, df_tsv.iloc[i].page+2), 7)
del i
i = 236-2
quotes[i] = QuoteInfo(
    df_tsv.iloc[i].page,
    "The Witch King stabs Frodo",
    (df_tsv.iloc[i].t_mid+150, df_tsv.iloc[i].page),
    (df_tsv.iloc[i].t_mid+800, df_tsv.iloc[i].page), 8)
del i
i = 245-2
quotes[i] = QuoteInfo(
    df_tsv.iloc[i].page,
    '"This is beyond my skill to heal."',
    (df_tsv.iloc[i].t_mid-110, df_tsv.iloc[i].page),
    (df_tsv.iloc[i].t_mid-4300, df_tsv.iloc[i].page-4), 7)
del i
i = 260-2
quotes[i].text = '"You are in the House of Elrond\nand it is ten o\'clock in the morning"'
del i
quotes[274-2].text = '"Bilbo!" "Hello, Frodo my lad!"'
i = 294-2
quotes[i] = QuoteInfo(
    df_tsv.iloc[i].page,
    '"Bring forth the Ring, Frodo."',
    (df_tsv.iloc[i].t_mid+150, df_tsv.iloc[i].page),
    (df_tsv.iloc[i].t_mid+800, df_tsv.iloc[i].page-4), 7)
del i
i = 302-2
quotes[i] = QuoteInfo(
    df_tsv.iloc[i].page,
    '"Ash nazg durbatulûk..."',
    (df_tsv.iloc[i].t_mid+150, df_tsv.iloc[i].page),
    (df_tsv.iloc[i].t_mid+800, df_tsv.iloc[i].page-4), 7)
del i
i = 311-2
quotes[i] = QuoteInfo(
    df_tsv.iloc[i].page,
    '"I gave you the chance of aiding me willingly.\nBut you have elected the way of pain!"',
    (df_tsv.iloc[i].t_mid-50, df_tsv.iloc[i].page-1),
    (df_tsv.iloc[i].t_mid-3000, df_tsv.iloc[i].page-6), 7)
del i
i = 322-2
quotes[i] = QuoteInfo(
    df_tsv.iloc[i].page,
    '"I will take the Ring to Mordor...\nthough I do not know the way."',
    (df_tsv.iloc[i].t_mid+150, df_tsv.iloc[i].page),
    (df_tsv.iloc[i].t_mid+800, df_tsv.iloc[i].page-4), 7)
del i
i = 334-2
quotes[i] = QuoteInfo(
    df_tsv.iloc[i].page,
    '"The Ringbearer is setting out\non the quest of Mount Doom"',
    (df_tsv.iloc[i].t_mid+150, df_tsv.iloc[i].page),
    (df_tsv.iloc[i].t_mid+800, df_tsv.iloc[i].page), 7)
del i
i = 344-2
quotes[i].text = '"This will be the death of the hobbits!"'
quotes[i].fontsize = 7
del i
i = 362-2
quotes[i] = QuoteInfo(
    df_tsv.iloc[i].page,
    '"Speak Friend and Enter"',
    (df_tsv.iloc[i].t_mid+150, df_tsv.iloc[i].page),
    (df_tsv.iloc[i].t_mid+800, df_tsv.iloc[i].page-4), 7)
del i
i = 374-2
quotes[i] = QuoteInfo(
    df_tsv.iloc[i].page,
    '"I\'ve no memory of this place."',
    (df_tsv.iloc[i].t_mid+150, df_tsv.iloc[i].page),
    (df_tsv.iloc[i].t_mid+800, df_tsv.iloc[i].page-4), 8)
del i
i = 390-2
quotes[i] = QuoteInfo(
    df_tsv.iloc[i].page,
    '"They have a cave troll."',
    (df_tsv.iloc[i].t_mid+150, df_tsv.iloc[i].page),
    (df_tsv.iloc[i].t_mid+800, df_tsv.iloc[i].page-4), 8)
del i
i = 402-2
quotes[i].text = '"You cannot pass!"'
quotes[i].xy = (df_tsv.iloc[i].t_mid+150, df_tsv.iloc[i].page)
quotes[i].xytext = (df_tsv.iloc[i].t_mid+800, df_tsv.iloc[i].page-3)
del i
i = 418-2
quotes[i].text = '"The dwarf breathes so loud,\nwe could have shot him in the dark."'
quotes[i].xy = (df_tsv.iloc[i].t_mid-150, df_tsv.iloc[i].page)
quotes[i].xytext = (df_tsv.iloc[i].t_mid-6000, df_tsv.iloc[i].page+3)
del i
i = 447-2
quotes[i].text = '"All shall love me, and despair!"'
quotes[i].xy = (df_tsv.iloc[i].t_mid-150, df_tsv.iloc[i].page)
quotes[i].xytext = (df_tsv.iloc[i].t_mid-5000, df_tsv.iloc[i].page-2)
del i
i = 460-2
quotes[i] = QuoteInfo(
    df_tsv.iloc[i].page,
    '"May it be a light for you in dark places,\nwhen all other lights go out."',
    (df_tsv.iloc[i].t_mid-150, df_tsv.iloc[i].page),
    (df_tsv.iloc[i].t_mid-6000, df_tsv.iloc[i].page), 7)
del i
i = 478-2
quotes[i].text = '"Frodo, the Argonath!"'
quotes[i].xy = (df_tsv.iloc[i].t_mid-150, df_tsv.iloc[i].page)
quotes[i].xytext = (df_tsv.iloc[i].t_mid-4000, df_tsv.iloc[i].page-2)
del i
i = 486-2
quotes[i] = QuoteInfo(
    df_tsv.iloc[i].page,
    '"It should be mine! Give it to me!"',
    (df_tsv.iloc[i].t_mid-150, df_tsv.iloc[i].page),
    (df_tsv.iloc[i].t_mid-5000, df_tsv.iloc[i].page+4), 7)
del i
i = 503-2
quotes[i] = QuoteInfo(
    df_tsv.iloc[i].page,
    '"The Horn of Gondor!"',
    (df_tsv.iloc[i].t_mid-150, df_tsv.iloc[i].page),
    (df_tsv.iloc[i].t_mid-3300, df_tsv.iloc[i].page+4), 7)
del i
i = 513-2
quotes[i] = QuoteInfo(
    df_tsv.iloc[i].page,
    '"Frodo\'s fate is no longer in our hands."',
    (df_tsv.iloc[i].t_mid-150, df_tsv.iloc[i].page),
    (df_tsv.iloc[i].t_mid-5600, df_tsv.iloc[i].page+4), 7)
del i
for i, quote in quotes.items():
    ax.annotate(
        quote.text,
        xy=quote.xy,
        xytext=quote.xytext,
        arrowprops=dict(
            width=0.2,
            headwidth=4,
            headlength=4,
            color='black',
    #         arrowstyle="-|>,head_width=0.4,head_length=0.8",
    #         shrinkA=0,
    #         shrinkB=0
        ),
        color='black',
        fontsize=quote.fontsize,
        xycoords='data',
        textcoords='data',
    #     transform = ax.transAxes,
    )
ax.set_ylabel("Page number of book")
ax.set_xlabel("Duration into movie (extended edition)");
ax.xaxis.set_label_position('top');
ax.xaxis.tick_top()
ax.xaxis.set_major_formatter(
     matplotlib.ticker.FuncFormatter(format_timedelta)
)
ax.set_yticks(chapter_page_numbers)
ax.set_ylim(1, 475)
ax.yaxis.set_inverted(True)
ax.set_title("The Fellowship of the Ring: book and movie");

In [ ]:
fig.canvas.draw();

In [ ]:
# Save to file
fig.savefig(
    "fellowship-of-the-ring.png",
    bbox_inches='tight',
    metadata = {"Title": "Fellowship of the Ring: adaptation graph", "Author": "Nathaniel M. Foley-Beaver"},
    dpi=250,
    facecolor="w", # white background
);
# Save to file
fig.savefig(
    "fellowship-of-the-ring.svg",
    bbox_inches='tight',
    metadata = {"Title": "Fellowship of the Ring: Adaptation Graph", "Creator": "Nathaniel M. Foley-Beaver"},
    facecolor="w", # white background
);

In [ ]:
plt.close(fig); del fig, ax;

TODO:

- [x] Add horizontal lines for chapter breaks
- [x] Add chapter labels
- [x] Add quotes
- [ ] Add publishing information (Ballantine Books, paperback, ISBN 0-345-33970-3 and 0-345-33971-1)

# Interactive HTML scatterplot

- https://codepointtech.com/create-interactive-scatter-plots-in-python-with-plotly/
- https://python-graph-gallery.com/511-interactive-scatterplot-with-plotly/
- https://stackoverflow.com/questions/59953431/how-to-change-plotly-figure-size

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
# import math
# def get_timedelta(x):
#     if math.isnan(x):
#         return None
#     else:
#         return datetime.timedelta(seconds=int(x))
# t_mid_delta = [get_timedelta(x) for x in t_mid]
# This won't work because plotly can't handle plotting timedelta directly

In [ ]:
book_quote = df_tsv['Book quote']
movie_quote = df_tsv['Movie quote / description']
df_plot = pd.DataFrame(data={
    'page_number':page_number,
    't_mid':t_mid,
    'book_quote': book_quote,
    'movie_quote': movie_quote,
})

In [ ]:
df_plot

In [ ]:
# fig = px.scatter(
#     df_plot,
#     x="t_mid",
#     y="page_number",
#     title="The Fellowship of the Ring: book and movie",
#     labels={'X_Value':'Independent Variable', 'Y_Value':'Dependent Variable'})
# fig.show()

In [ ]:
# del fig

In [ ]:
# fig = go.Figure(
#     data=go.Scatter(
#     # x=df_plot['t_mid'],
#     x=pd.to_datetime(df_plot['t_mid'],unit='s'),
#     y=df_plot['page_number'],
#     mode='markers',
#     marker=dict(
#         size=10,
#         # color="forestgreen",
#         color='rgba(0,0,0,0)',  # Transparent fill
#         line=dict(
#             width=2,
#             color='forestgreen'  # Visible border
#         )
#     ),
    
#     # Custom hover template
#     hovertemplate=
#         "Movie timestamp (extended edition): %{x}<br>" +
#         "Page number: %{y}<br>" +
#         "Book: %{text2}<br>" +
#         "Movie: %{text}<extra></extra>", # <extra></extra> removes default trace name
#     text=df_plot['movie_quote'],
#     # text2=df_plot['book_quote'],
# ))

# fig.update_layout(
#     title="The Fellowship of the Ring: book and movie",
#     yaxis_title="Page number of book",
#     xaxis_title="Duration into movie (extended edition)",
#     autosize=False,
#     width=6*150,
#     height=13*150,
#     yaxis_autorange = "reversed",
#     hoverlabel=dict(
#         bgcolor="white",
#         font_size=16,
#         font_family="Rockwell"
#     )
# )
# fig.show()

In [ ]:
# del fig

In [ ]:
fig = px.scatter(
    df_plot,
    # x="t_mid",
    x=pd.to_datetime(df_plot['t_mid'],unit='s'),
    y="page_number",
    hover_data=["page_number", 'book_quote', 'movie_quote'],
)
fig.update_traces(
    marker=dict(
        size=10,
        # color="forestgreen",
        color='rgba(0,0,0,0)',  # Transparent fill
        line=dict(
            width=2,
            color='forestgreen'  # Visible border
        )
    ),
)
fig.update_layout(
    title="The Fellowship of the Ring: book and movie",
    yaxis_title="Page number of book",
    xaxis_title="Duration into movie (extended edition)",
    autosize=False,
    width=6*150,
    height=13*150,
    yaxis_autorange = "reversed",
    hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="Rockwell"
    ),
    xaxis_tickformat="%H:%M:%S",
    xaxis=dict(
        tickformat='%H:%M:%S' # Forces HH:MM:SS format
    )
)
fig.show()

In [ ]:
fig.write_html("fellowship-of-the-ring-interactive.html")